In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [ ]:
import pandas as pd
from pathlib import Path

# Path to main folder
path_to_main_folder = Path("/home/gzu5140/Keerthana_b1042/grnInference/analysisData/parameter_scan_1_20/A_B_2_states")

# Find all CSV files inside subfolders (recursive search)
csv_files = list(path_to_main_folder.rglob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

# Read and concat
df_list = [pd.read_csv(f) for f in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
combined_df_param = combined_df

In [ ]:
combined_df[combined_df['param_index']=="0_1_2"]

In [ ]:
combined_df.sort_values(by = "param_index")

In [ ]:
import matplotlib.pyplot as plt
true_positive = []
p_val = np.logspace(np.log10(0.0001), np.log10(1), 50)
for p in p_val:
    combined_df_filter_1 = combined_df_param[combined_df_param['pval_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20']<p]
    true_positive.append(len(combined_df_filter_1)/len(combined_df_param))

plt.plot(p_val, true_positive, "x")
plt.xlabel("p-value threshold")
plt.xscale('log')
plt.title("k_off is within range")
plt.ylabel("Fraction of true positives")
plt.axvline(0.01, linestyle = "--")
plt.show()

In [ ]:
parameter_df = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv", index_col=0)


In [ ]:
parameter_df

In [ ]:
parameter_df['gene_id'].unique()

In [ ]:
parameter_df_filtered = parameter_df[parameter_df['gene_id'] != 3]
parameter_df_filtered

In [ ]:
# First filter on k_off
parameter_df_filtered = parameter_df[parameter_df['gene_id'] != 3]
parameter_df_filtered = parameter_df_filtered[(parameter_df_filtered["k_off"] > 0.06) & (parameter_df_filtered["k_off"] < 55)]

# Keep only pair_ids where all three unique gene_ids are present
valid_pairs = (
    parameter_df_filtered.groupby("pair_id")["gene_id"]
    .nunique()                           # count unique gene_ids per pair_id
    .loc[lambda x: x == 2]               # keep only those with 3 genes
    .index
)

parameter_df_filtered = parameter_df_filtered[parameter_df_filtered["pair_id"].isin(valid_pairs)]

parameter_df_filtered


In [ ]:
# Function to generate param_index directly from pair_id
def make_param_index(pid):
    start = pid * 3
    return f"{start}_{start+1}_{start+2}"

parameter_df_filtered["param_index"] = parameter_df_filtered["pair_id"].apply(make_param_index)

parameter_df_filtered


In [ ]:
combined_df_param = pd.merge(combined_df, parameter_df_filtered, on="param_index")
combined_df_param

In [ ]:
import matplotlib.pyplot as plt
true_positive = []
p = 0.01
combined_df_param = combined_df_param[combined_df_param['pval_gene_gene_gene_1_mRNA_gene_2_mRNA_t1_t20']<p]
z_score = np.arange(0, 101, 5)
for z in z_score:
    combined_df_filter_2 = combined_df_filter_1[abs(combined_df_filter_1['zscore_twin_vs_random_gene_1_mRNA_gene_2_mRNA_t1'])<z]
    true_positive.append(len(combined_df_filter_2)/len(combined_df_filter_1))

plt.plot(z_score, true_positive, "x")
plt.xlabel("Z-score threshold")
plt.ylabel("Fraction of true negatives")
plt.axvline(0.01, linestyle = "--")
plt.title("Detection of multiple-states: Fraction of parameters with Z_score < threshold")
plt.show()

In [ ]:
z = 10
combined_df_filter_2 = combined_df_filter_1[abs(combined_df_filter_1['zscore_twin_vs_random_gene_1_mRNA_gene_2_mRNA_t1'])<z]
true_positive = []
false_positive = []
p_val = np.logspace(np.log10(0.000001), np.log10(1), 50)
for p in p_val:
    A_to_B = combined_df_filter_2[abs(combined_df_filter_2['directed_pval_gene_1_mRNA_t1__gene_2_mRNA_t20'])<p]
    true_positive.append(len(A_to_B)/len(combined_df_filter_2))
    B_to_A = combined_df_filter_2[abs(combined_df_filter_2['directed_pval_gene_2_mRNA_t1__gene_1_mRNA_t20'])<p]
    false_positive.append(len(B_to_A)/len(combined_df_filter_2))
plt.plot(p_val, true_positive, c = "black", label = "True Positive  (A -> B is significant)")
plt.plot(p_val, false_positive, c = "red", label = "False Positive fraction (B -> A is significant)")
plt.xlabel("P-value thresholds")
plt.ylabel("Fraction")
plt.axvline(0.01, linestyle = "--")
plt.xscale('log')
plt.legend()
plt.show()

plt.plot(true_positive, false_positive)
plt.xlabel("True Positive fraction ")
plt.ylabel("False Positive fraction")
plt.xlim(0,1)
plt.ylim(0,1)
plt.show()